# Pricing Project

In [13]:
# Problem 1: MNL Model

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import logsumexp

df = pd.read_csv('Data/data.csv')

feature_cols = [
    'prop_starrating', 'prop_review_score', 'prop_brand_bool',
    'prop_location_score', 'prop_accesibility_score',
    'prop_log_historical_price', 'price_usd', 'promotion_flag'
]

# Normalize features
means = df[feature_cols].mean()
stds = df[feature_cols].std()
estimation_df = df.copy()
estimation_df[feature_cols] = (df[feature_cols] - means) / stds

# Group by search session
groups = []
for _, g in estimation_df.groupby('srch_id'):
    groups.append((g[feature_cols].values, g['booking_bool'].values))

def neg_ll(beta):
    b0, b = beta[0], beta[1:]
    nll, grad = 0.0, np.zeros_like(beta)
    for X, y in groups:
        u = b0 + X @ b
        ld = logsumexp(np.append(0.0, u))
        p = np.exp(u - ld)
        d = p - y
        nll += ld - y @ u
        grad[0] += d.sum()
        grad[1:] += X.T @ d
    return nll, grad

result = minimize(neg_ll, x0=np.zeros(9), jac=True, method='L-BFGS-B')
beta_star = result.x

print('Estimated MNL coefficients (standardized features):')
for name, val in zip(['intercept'] + feature_cols, beta_star):
    print(f'  {name:35s} {val:+f}')

# Raw (unstandardized) coefficients
b_raw_feat = beta_star[1:] / stds[feature_cols].values
b0_raw = beta_star[0] - (beta_star[1:] * means[feature_cols].values / stds[feature_cols].values).sum()
print('\nRaw MNL coefficients (original feature units):')
for name, val in zip(['intercept'] + feature_cols, [b0_raw] + b_raw_feat.tolist()):
    print(f'  {name:35s} {val:+.8f}')
print(f'\nFinal log-likelihood: {-result.fun:.2f}')

Estimated MNL coefficients (standardized features):
  intercept                           -1.746680
  prop_starrating                     +0.408132
  prop_review_score                   +0.108754
  prop_brand_bool                     +0.101381
  prop_location_score                 +0.021995
  prop_accesibility_score             +0.043445
  prop_log_historical_price           -0.066872
  price_usd                           -1.331110
  promotion_flag                      +0.159765

Raw MNL coefficients (original feature units):
  intercept                           -2.81529490
  prop_starrating                     +0.47613042
  prop_review_score                   +0.11988869
  prop_brand_bool                     +0.22991630
  prop_location_score                 +0.01631512
  prop_accesibility_score             +0.56288751
  prop_log_historical_price           -0.03736903
  price_usd                           -0.00732305
  promotion_flag                      +0.45399470

Final log-likelih

In [14]:
# Problem 2: Assortment Optimization under MNL

def compute_v_price(df_s):
    X_norm = (df_s[feature_cols] - means) / stds
    u = beta_star[0] + X_norm.to_numpy() @ beta_star[1:]
    return np.exp(u), df_s['price_usd'].to_numpy()

def optimal_assortment(v, price):
    n = len(price)
    order = np.argsort(-price, kind='mergesort')
    p_s, v_s = price[order], v[order]
    cum_v = np.cumsum(v_s)
    cum_pv = np.cumsum(p_s * v_s)
    cuts = np.flatnonzero(np.r_[p_s[1:] != p_s[:-1], True])
    best_rev, best_k = 0.0, -1
    for k in cuts:
        rev = cum_pv[k] / (1.0 + cum_v[k])
        if rev > best_rev:
            best_rev, best_k = float(rev), int(k)
    m_s = np.zeros(n, dtype=bool)
    if best_k >= 0:
        m_s[:best_k + 1] = True
    mask = np.zeros(n, dtype=bool)
    mask[order] = m_s
    return mask, best_rev

results = []
for name in ['data1', 'data2', 'data3', 'data4']:
    df_s = pd.read_csv(f'Data/{name}.csv')
    v, price = compute_v_price(df_s)
    mask, rev = optimal_assortment(v, price)

    results.append({
        'dataset': name,
        'n_hotels': int(len(df_s)),
        'n_offered': int(mask.sum()),
        'expected_revenue': rev,
    })

    offered_idx = np.flatnonzero(mask)
    offered_prices = price[mask]

    print(f'\n{name}: optimal offered subset size = {mask.sum()} out of {len(df_s)}')
    print(f'{name}: expected revenue = {rev:.6f}')
    print(f'{name}: offered row indices (0-based) = {offered_idx.tolist()}')
    print(f'{name}: offered prices = {np.round(np.sort(offered_prices), 2).tolist()}')

summary_df = pd.DataFrame(results).sort_values('dataset')
print('\nSummary')
display(summary_df)


data1: optimal offered subset size = 18 out of 27
data1: expected revenue = 107.352012
data1: offered row indices (0-based) = [0, 1, 2, 3, 4, 5, 6, 12, 15, 17, 18, 19, 20, 21, 22, 23, 24, 26]
data1: offered prices = [108, 112, 118, 120, 121, 121, 125, 131, 131, 140, 144, 145, 147, 147, 150, 154, 156, 158]

data2: optimal offered subset size = 10 out of 29
data2: expected revenue = 131.110898
data2: offered row indices (0-based) = [0, 1, 6, 7, 8, 9, 10, 21, 23, 25]
data2: offered prices = [133, 134, 137, 143, 146, 147, 161, 206, 212, 233]

data3: optimal offered subset size = 18 out of 26
data3: expected revenue = 121.053221
data3: offered row indices (0-based) = [0, 1, 2, 3, 4, 5, 7, 8, 10, 11, 13, 14, 15, 16, 18, 19, 23, 24]
data3: offered prices = [123, 128, 129, 138, 138, 140, 144, 150, 153, 171, 180, 181, 190, 191, 195, 211, 281, 603]

data4: optimal offered subset size = 11 out of 27
data4: expected revenue = 97.407947
data4: offered row indices (0-based) = [3, 4, 6, 8, 10, 15, 1

,dataset,n_hotels,n_offered,expected_revenue
0,data1,27,18,107.352012
1,data2,29,10,131.110898
2,data3,26,18,121.053221
3,data4,27,11,97.407947


In [15]:
# Problem 3: Pricing under MNL
# Display all hotels and choose prices to maximize expected revenue.

price_col = 'price_usd'
price_idx = feature_cols.index(price_col)

b0 = float(beta_star[0])
b = beta_star[1:].astype(float)
b_price = float(b[price_idx])
mu = float(means[price_col])
sd = float(stds[price_col])

def optimize_prices(df_s, lo, hi):
    X = df_s[feature_cols].to_numpy(dtype=float)
    z = (X - means[feature_cols].to_numpy()) / stds[feature_cols].to_numpy()
    z_np = z.copy()
    z_np[:, price_idx] = 0.0
    base = b0 + z_np @ b

    def neg_rev(p):
        u = base + b_price * ((p - mu) / sd)
        v = np.exp(u)
        D = 1.0 + v.sum()
        N = float((p * v).sum())
        c = b_price / sd
        dv = c * v
        dN = v + p * dv
        dD = dv
        grad = (dN * D - N * dD) / (D * D)
        return -N / D, -grad

    p0 = df_s[price_col].to_numpy(dtype=float)
    bounds = [(lo, hi)] * len(p0)
    res = minimize(neg_rev, p0, jac=True, bounds=bounds, method='L-BFGS-B')
    p_star = res.x.astype(float)

    u0 = base + b_price * ((p0 - mu) / sd)
    v0 = np.exp(u0)
    rev0 = float((p0 * v0).sum() / (1.0 + v0.sum()))

    u_s = base + b_price * ((p_star - mu) / sd)
    v_s = np.exp(u_s)
    rev_s = float((p_star * v_s).sum() / (1.0 + v_s.sum()))

    return p0, p_star, rev0, rev_s, res.message, int(res.nit)

pricing_results = []
for name in ['data1', 'data2', 'data3', 'data4']:
    df_s = pd.read_csv(f'Data/{name}.csv')
    p_now = df_s[price_col].to_numpy(dtype=float)
    lo, hi = 1.0, max(1000.0, 3.0 * float(np.max(p_now)))
    p0, p_star, rev0, rev_s, msg, n_it = optimize_prices(df_s, lo, hi)

    pricing_results.append({
        'dataset': name,
        'n_hotels': int(len(df_s)),
        'rev_current': rev0,
        'rev_opt': rev_s,
        'rev_lift': rev_s - rev0,
        'price_min_current': float(np.min(p0)),
        'price_max_current': float(np.max(p0)),
        'price_min_opt': float(np.min(p_star)),
        'price_max_opt': float(np.max(p_star)),
        'iters': n_it,
    })

    print(f'\n{name}: expected revenue (current prices) = {rev0:.6f}')
    print(f'{name}: expected revenue (optimal prices) = {rev_s:.6f}')
    print(f'{name}: optimizer status = {msg}')
    print(f'{name}: optimal prices (rounded) = {np.round(p_star, 2).tolist()}')

pricing_summary = pd.DataFrame(pricing_results).sort_values('dataset')
print('\nPricing summary')
display(pricing_summary)


data1: expected revenue (current prices) = 104.648936
data1: expected revenue (optimal prices) = 177.802410
data1: optimizer status = CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL
data1: optimal prices (rounded) = [314.33, 314.35, 314.35, 314.34, 314.34, 314.35, 314.35, 314.38, 314.24, 314.42, 314.36, 314.38, 314.41, 314.39, 314.37, 314.35, 314.39, 314.36, 314.36, 314.36, 314.37, 314.37, 314.36, 314.35, 314.36, 314.36, 314.36]

data2: expected revenue (current prices) = 113.358769
data2: expected revenue (optimal prices) = 248.796037
data2: optimizer status = CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL
data2: optimal prices (rounded) = [385.32, 385.37, 385.35, 385.34, 385.35, 385.36, 385.36, 385.35, 385.36, 385.35, 385.36, 385.36, 385.36, 385.35, 385.35, 385.3, 385.34, 385.36, 385.33, 385.34, 385.35, 385.35, 385.35, 385.35, 385.33, 385.35, 385.34, 385.35, 385.33]

data3: expected revenue (current prices) = 118.166957
data3: expected revenue (optimal prices) = 176.469420
data3:

,dataset,n_hotels,rev_current,rev_opt,rev_lift,price_min_current,price_max_current,price_min_opt,price_max_opt,iters
0,data1,27,104.648936,177.802410,73.153474,75.0,158.0,314.237661,314.421413,10
1,data2,29,113.358769,248.796037,135.437268,69.0,233.0,385.303821,385.370573,14
2,data3,26,118.166957,176.469420,58.302464,101.0,603.0,312.973833,313.040916,14
3,data4,27,78.688792,214.469823,135.781032,40.0,195.0,350.999419,351.077285,14


In [16]:
# Problem 4: Mixture of MNL (MMNL) - customer types by booking window
# Type 1 = early (booking window >= 7 days), Type 2 = late (booking window < 7 days).

query_window = df.groupby('srch_id')['srch_booking_window'].first()
early_ids = set(query_window.index[query_window >= 7])
late_ids  = set(query_window.index[query_window < 7])

n_queries = len(query_window)
theta1 = len(early_ids) / n_queries   # early
theta2 = len(late_ids)  / n_queries   # late
print(f'theta1 (early, window >= 7) = {theta1:.4f}   ({len(early_ids)} queries)')
print(f'theta2 (late,  window <  7) = {theta2:.4f}   ({len(late_ids)} queries)')
assert abs(theta1 + theta2 - 1.0) < 1e-9

groups_type1 = []   # early
groups_type2 = []   # late
for sid, g in estimation_df.groupby('srch_id'):
    item = (g[feature_cols].values, g['booking_bool'].values)
    if sid in early_ids:
        groups_type1.append(item)
    else:
        groups_type2.append(item)

def neg_ll_groups(beta, groups_k):
    b0, b = beta[0], beta[1:]
    nll, grad = 0.0, np.zeros_like(beta)
    for X, y in groups_k:
        u = b0 + X @ b
        ld = logsumexp(np.append(0.0, u))
        p = np.exp(u - ld)
        d = p - y
        nll += ld - y @ u
        grad[0] += d.sum()
        grad[1:] += X.T @ d
    return nll, grad

res1 = minimize(lambda b: neg_ll_groups(b, groups_type1),
                x0=np.zeros(9), jac=True, method='L-BFGS-B')
res2 = minimize(lambda b: neg_ll_groups(b, groups_type2),
                x0=np.zeros(9), jac=True, method='L-BFGS-B')

beta_star_type1 = res1.x   # early 
beta_star_type2 = res2.x   # late  

param_names = ['intercept'] + feature_cols
print('\nMMNL coefficients (standardized features):')
print(f"{'parameter':35s} {'type1 (early)':>15s} {'type2 (late)':>15s} {'diff (late-early)':>20s}")
for name, b1, b2 in zip(param_names, beta_star_type1, beta_star_type2):
    print(f'  {name:33s} {b1:+14.6f} {b2:+14.6f} {b2-b1:+19.6f}')


theta1 (early, window >= 7) = 0.5431   (4537 queries)
theta2 (late,  window <  7) = 0.4569   (3817 queries)

MMNL coefficients (standardized features):
parameter                             type1 (early)    type2 (late)    diff (late-early)
  intercept                              -1.918048      -1.539647           +0.378401
  prop_starrating                        +0.379045      +0.462757           +0.083712
  prop_review_score                      +0.124625      +0.095587           -0.029038
  prop_brand_bool                        +0.092095      +0.112904           +0.020808
  prop_location_score                    -0.021460      +0.087282           +0.108742
  prop_accesibility_score                +0.057844      +0.025559           -0.032284
  prop_log_historical_price              -0.094880      -0.031978           +0.062902
  price_usd                              -1.071570      -1.696916           -0.625346
  promotion_flag                         +0.134803      +0.194144      

In [17]:
# Problem 5: Early vs. Late Reservations 
# S  : optimal assortment, type unknown — MMNL via MILP
# S1 : optimal assortment for type 1 (early) — revenue-ordered
# S2 : optimal assortment for type 2 (late)  — revenue-ordered

import gurobipy as gp
from gurobipy import GRB

def compute_v(df_s, beta_vec):
    X_norm = (df_s[feature_cols] - means) / stds
    u = beta_vec[0] + X_norm.to_numpy() @ beta_vec[1:]
    return np.exp(u)

def mnl_revenue(v, price, mask):
    vS, pS = v[mask], price[mask]
    return float((pS * vS).sum() / (1.0 + vS.sum()))

def revenue_ordered_optimal(v, price):
    n = len(price)
    order = np.argsort(-price, kind='mergesort')
    p_s, v_s = price[order], v[order]
    cum_v = np.cumsum(v_s)
    cum_pv = np.cumsum(p_s * v_s)
    cuts = np.flatnonzero(np.r_[p_s[1:] != p_s[:-1], True])
    best_rev, best_k = 0.0, -1
    for k in cuts:
        rev = cum_pv[k] / (1.0 + cum_v[k])
        if rev > best_rev:
            best_rev, best_k = float(rev), int(k)
    m_s = np.zeros(n, dtype=bool)
    if best_k >= 0:
        m_s[:best_k + 1] = True
    mask = np.zeros(n, dtype=bool)
    mask[order] = m_s
    return mask, best_rev

def solve_mmnl_milp(v_by_type, price, theta_by_type, time_limit=60):
    K, n = len(v_by_type), len(price)
    m = gp.Model('mmnl_assortment')
    m.setParam('OutputFlag', 0)
    m.setParam('TimeLimit', time_limit)

    x = m.addVars(n, vtype=GRB.BINARY, name='x')
    y = m.addVars(K, lb=0.0, ub=1.0, name='y')
    w = m.addVars(n, K, lb=0.0, ub=1.0, name='w')

    for j in range(n):
        for k in range(K):
            m.addConstr(w[j, k] <= x[j])
            m.addConstr(w[j, k] <= y[k])
            m.addConstr(w[j, k] >= y[k] - (1 - x[j]))

    for k in range(K):
        m.addConstr(y[k] + gp.quicksum(float(v_by_type[k][j]) * w[j, k] for j in range(n)) == 1.0)

    m.setObjective(
        gp.quicksum(
            theta_by_type[k] * float(price[j]) * float(v_by_type[k][j]) * w[j, k]
            for k in range(K) for j in range(n)
        ),
        GRB.MAXIMIZE,
    )
    m.optimize()
    status_map = {GRB.OPTIMAL: 'OPTIMAL', GRB.TIME_LIMIT: 'TIME_LIMIT',
                  GRB.INFEASIBLE: 'INFEASIBLE', GRB.INF_OR_UNBD: 'INF_OR_UNBD'}
    status_str = status_map.get(m.Status, f'code={m.Status}')
    gap = m.MIPGap if m.Status in (GRB.OPTIMAL, GRB.TIME_LIMIT) else float('nan')
    print(f'    MILP: status={status_str}, MIPGap={gap:.2e}')
    mask = np.array([x[j].X > 0.5 for j in range(n)], dtype=bool)
    return mask, float(m.ObjVal)

rows = []
for name in ['data1', 'data2', 'data3', 'data4']:
    df_s = pd.read_csv(f'Data/{name}.csv')
    price = df_s['price_usd'].to_numpy(dtype=float)
    v1 = compute_v(df_s, beta_star_type1)
    v2 = compute_v(df_s, beta_star_type2)

    S_mask,  S_rev = solve_mmnl_milp([v1, v2], price, [theta1, theta2])
    S1_mask, _ = revenue_ordered_optimal(v1, price)
    S2_mask, _ = revenue_ordered_optimal(v2, price)

    R1_S, R1_S1 = mnl_revenue(v1, price, S_mask), mnl_revenue(v1, price, S1_mask)
    R2_S, R2_S2 = mnl_revenue(v2, price, S_mask), mnl_revenue(v2, price, S2_mask)

    print(f'\n--- {name} (n={len(df_s)}) ---')
    print(f'  Type unknown : |S|  = {S_mask.sum():2d}   mixture revenue = {S_rev:7.4f}')
    print(f'  Type 1       : |S1| = {S1_mask.sum():2d}   R1(S1) = {R1_S1:7.4f}   R1(S) = {R1_S:7.4f}   gap = {R1_S1 - R1_S:+.4f}')
    print(f'  Type 2       : |S2| = {S2_mask.sum():2d}   R2(S2) = {R2_S2:7.4f}   R2(S) = {R2_S:7.4f}   gap = {R2_S2 - R2_S:+.4f}')

    print(f'  S  hotels (type unknown): {list(map(int, np.where(S_mask)[0]))}')
    print(f'  S1 hotels (early): {list(map(int, np.where(S1_mask)[0]))}')
    print(f'  S2 hotels (late): {list(map(int, np.where(S2_mask)[0]))}')
    
    rows.append({
        'dataset': name,
        '|S|': int(S_mask.sum()),   'R_mix(S)': round(S_rev, 2),
        '|S1|': int(S1_mask.sum()), 'R1(S1)': round(R1_S1, 2), 'R1(S)': round(R1_S, 2), 'gap1': round(R1_S1 - R1_S, 4),
        '|S2|': int(S2_mask.sum()), 'R2(S2)': round(R2_S2, 2), 'R2(S)': round(R2_S, 2), 'gap2': round(R2_S2 - R2_S, 4),
    })

summary5 = pd.DataFrame(rows)
print('\nProblem 5 summary')
display(summary5)

    MILP: status=OPTIMAL, MIPGap=0.00e+00

--- data1 (n=27) ---
  Type unknown : |S|  = 18   mixture revenue = 107.3047
  Type 1       : |S1| = 21   R1(S1) = 103.4705   R1(S) = 103.2525   gap = +0.2180
  Type 2       : |S2| = 16   R2(S2) = 112.2233   R2(S) = 112.1212   gap = +0.1020
  S  hotels (type unknown): [0, 1, 2, 3, 4, 5, 6, 12, 15, 17, 18, 19, 20, 21, 22, 23, 24, 26]
  S1 hotels (early): [0, 1, 2, 3, 4, 5, 6, 9, 12, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 26]
  S2 hotels (late): [0, 1, 2, 3, 4, 5, 6, 15, 17, 18, 19, 20, 21, 22, 23, 24]
    MILP: status=OPTIMAL, MIPGap=0.00e+00

--- data2 (n=29) ---
  Type unknown : |S|  = 10   mixture revenue = 131.2788
  Type 1       : |S1| = 10   R1(S1) = 126.9221   R1(S) = 126.9221   gap = +0.0000
  Type 2       : |S2| =  7   R2(S2) = 137.3762   R2(S) = 136.4573   gap = +0.9189
  S  hotels (type unknown): [0, 1, 6, 7, 8, 9, 10, 21, 23, 25]
  S1 hotels (early): [0, 1, 6, 7, 8, 9, 10, 21, 23, 25]
  S2 hotels (late): [0, 1, 6, 7, 8, 10, 21]

,dataset,|S|,R_mix(S),|S1|,R1(S1),R1(S),gap1,|S2|,R2(S2),R2(S),gap2
0,data1,18,107.30,21,103.47,103.25,0.2180,16,112.22,112.12,0.1020
1,data2,10,131.28,10,126.92,126.92,0.0000,7,137.38,136.46,0.9189
2,data3,18,121.00,19,117.52,117.44,0.0827,17,125.35,125.23,0.1195
3,data4,11,97.38,11,94.69,94.69,0.0000,10,100.64,100.58,0.0623


In [18]:
# Problem 6: MMNL with a new customer type split - solo vs group traveler
# Type 1 = solo (srch_adults_count == 1), Type 2 = group (srch_adults_count > 1).

query_adults = df.groupby('srch_id')['srch_adults_count'].first()
solo_ids  = set(query_adults.index[query_adults == 1])
group_ids = set(query_adults.index[query_adults > 1])

n_queries   = len(query_adults)
theta_solo  = len(solo_ids)  / n_queries
theta_group = len(group_ids) / n_queries
print(f'theta_solo  (adults == 1) = {theta_solo:.4f}   ({len(solo_ids)} queries)')
print(f'theta_group (adults >  1) = {theta_group:.4f}   ({len(group_ids)} queries)')
assert abs(theta_solo + theta_group - 1.0) < 1e-9

groups_solo, groups_group = [], []
for sid, g in estimation_df.groupby('srch_id'):
    item = (g[feature_cols].values, g['booking_bool'].values)
    if sid in solo_ids:
        groups_solo.append(item)
    else:
        groups_group.append(item)

res_solo  = minimize(lambda b: neg_ll_groups(b, groups_solo),
                     x0=np.zeros(9), jac=True, method='L-BFGS-B')
res_group = minimize(lambda b: neg_ll_groups(b, groups_group),
                     x0=np.zeros(9), jac=True, method='L-BFGS-B')

beta_star_solo  = res_solo.x
beta_star_group = res_group.x


theta_solo  (adults == 1) = 0.2550   (2130 queries)
theta_group (adults >  1) = 0.7450   (6224 queries)


In [19]:
# Problem 6 (continued): repeat Problem 5 assortment analysis with the solo/group MMNL.
# Reuses solve_mmnl_milp, revenue_ordered_optimal, compute_v from Problem 5.

rows_p6 = []
for name in ['data1', 'data2', 'data3', 'data4']:
    df_s = pd.read_csv(f'Data/{name}.csv')
    price = df_s['price_usd'].to_numpy(dtype=float)
    v_s = compute_v(df_s, beta_star_solo)
    v_g = compute_v(df_s, beta_star_group)

    S_mask,  S_rev = solve_mmnl_milp([v_s, v_g], price, [theta_solo, theta_group])
    Ss_mask, _ = revenue_ordered_optimal(v_s, price)
    Sg_mask, _ = revenue_ordered_optimal(v_g, price)

    Rs_S, Rs_Ss = mnl_revenue(v_s, price, S_mask), mnl_revenue(v_s, price, Ss_mask)
    Rg_S, Rg_Sg = mnl_revenue(v_g, price, S_mask), mnl_revenue(v_g, price, Sg_mask)

    print(f'\n--- {name} (n={len(df_s)}) ---')
    print(f'  Type unknown : |S|  = {S_mask.sum():2d}   mixture revenue = {S_rev:7.4f}')
    print(f'  Solo         : |Ss| = {Ss_mask.sum():2d}   Rs(Ss) = {Rs_Ss:7.4f}   Rs(S) = {Rs_S:7.4f}   gap = {Rs_Ss - Rs_S:+.4f}')
    print(f'  Group        : |Sg| = {Sg_mask.sum():2d}   Rg(Sg) = {Rg_Sg:7.4f}   Rg(S) = {Rg_S:7.4f}   gap = {Rg_Sg - Rg_S:+.4f}')

    print(f'  S  hotels (type unknown): {list(map(int, np.where(S_mask)[0]))}')
    print(f'  Ss hotels (solo): {list(map(int, np.where(Ss_mask)[0]))}')
    print(f'  Sg hotels (group): {list(map(int, np.where(Sg_mask)[0]))}')

    rows_p6.append({
        'dataset': name,
        '|S|':  int(S_mask.sum()),   'R_mix(S)': round(S_rev, 2),
        '|Ss|': int(Ss_mask.sum()), 'Rs(Ss)': round(Rs_Ss, 2), 'Rs(S)': round(Rs_S, 2), 'gap_s': round(Rs_Ss - Rs_S, 4),
        '|Sg|': int(Sg_mask.sum()), 'Rg(Sg)': round(Rg_Sg, 2), 'Rg(S)': round(Rg_S, 2), 'gap_g': round(Rg_Sg - Rg_S, 4),
    })

summary6 = pd.DataFrame(rows_p6)
print('\nProblem 6 summary')
display(summary6)

    MILP: status=OPTIMAL, MIPGap=0.00e+00

--- data1 (n=27) ---
  Type unknown : |S|  = 18   mixture revenue = 107.2287
  Solo         : |Ss| = 16   Rs(Ss) = 114.9367   Rs(S) = 114.5511   gap = +0.3856
  Group        : |Sg| = 21   Rg(Sg) = 104.8383   Rg(S) = 104.7228   gap = +0.1155
  S  hotels (type unknown): [0, 1, 2, 3, 4, 5, 6, 12, 15, 17, 18, 19, 20, 21, 22, 23, 24, 26]
  Ss hotels (solo): [0, 1, 2, 3, 4, 5, 6, 15, 17, 18, 19, 20, 21, 22, 23, 24]
  Sg hotels (group): [0, 1, 2, 3, 4, 5, 6, 9, 12, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 26]
    MILP: status=OPTIMAL, MIPGap=0.00e+00

--- data2 (n=29) ---
  Type unknown : |S|  = 10   mixture revenue = 130.3880
  Solo         : |Ss| =  6   Rs(Ss) = 144.3979   Rs(S) = 141.0338   gap = +3.3640
  Group        : |Sg| = 10   Rg(Sg) = 126.7447   Rg(S) = 126.7447   gap = +0.0000
  S  hotels (type unknown): [0, 1, 6, 7, 8, 9, 10, 21, 23, 25]
  Ss hotels (solo): [0, 1, 6, 7, 8, 21]
  Sg hotels (group): [0, 1, 6, 7, 8, 9, 10, 21, 23, 25]
   

,dataset,|S|,R_mix(S),|Ss|,Rs(Ss),Rs(S),gap_s,|Sg|,Rg(Sg),Rg(S),gap_g
0,data1,18,107.23,16,114.94,114.55,0.3856,21,104.84,104.72,0.1155
1,data2,10,130.39,6,144.40,141.03,3.3640,10,126.74,126.74,0.0000
2,data3,18,120.78,15,132.40,131.64,0.7617,19,117.17,117.07,0.0962
3,data4,11,97.26,10,102.29,102.06,0.2354,11,95.62,95.62,0.0000


In [20]:
# Problem 7: AI agents as customers

df_agent = pd.read_csv('Data/data_empty_bool.csv')

feature_cols = [
    'prop_starrating', 'prop_review_score', 'prop_brand_bool',
    'prop_location_score', 'prop_accesibility_score',
    'prop_log_historical_price', 'price_usd', 'promotion_flag'
]

# Normalize features
means_agent = df_agent[feature_cols].mean()
stds_agent = df_agent[feature_cols].std()
estimation_df_agent = df_agent.copy()
estimation_df_agent[feature_cols] = (df_agent[feature_cols] - means_agent) / stds_agent

# Group by search session
groups_agent = []
for _, g in estimation_df_agent.groupby('srch_id'):
    groups_agent.append((g[feature_cols].values, g['booking_bool'].values))

def neg_ll_agent(beta):
    b0, b = beta[0], beta[1:]
    nll, grad = 0.0, np.zeros_like(beta)
    for X, y in groups_agent:
        u = b0 + X @ b
        ld = logsumexp(np.append(0.0, u))
        p = np.exp(u - ld)
        d = p - y
        nll += ld - y @ u
        grad[0] += d.sum()
        grad[1:] += X.T @ d
    return nll, grad

result_agent = minimize(neg_ll_agent, x0=np.zeros(9), jac=True, method='L-BFGS-B')
beta_star_agent = result_agent.x

print('Reestimated MNL coefficients on agent created dataset:')
for name, val in zip(['intercept'] + feature_cols, beta_star_agent):
    print(f'  {name:35s} {val:+f}')


Reestimated MNL coefficients on agent created dataset:
  intercept                           -5.778389
  prop_starrating                     +0.315486
  prop_review_score                   +0.717503
  prop_brand_bool                     +0.157374
  prop_location_score                 +0.074431
  prop_accesibility_score             +0.102504
  prop_log_historical_price           +0.000662
  price_usd                           -0.981444
  promotion_flag                      +0.529794


In [21]:
# Problem 7 (continued): Predictive evaluation
# Re-estimate Problem 1 MNL on full data
df_full = pd.read_csv('Data/data.csv')

feature_cols = [
    'prop_starrating', 'prop_review_score', 'prop_brand_bool',
    'prop_location_score', 'prop_accesibility_score',
    'prop_log_historical_price', 'price_usd', 'promotion_flag'
]

means_p1 = df_full[feature_cols].mean()
stds_p1 = df_full[feature_cols].std()
df_full[feature_cols] = (df_full[feature_cols] - means_p1) / stds_p1

groups_p1 = []
for _, g in df_full.groupby('srch_id'):
    groups_p1.append((g[feature_cols].values, g['booking_bool'].values))

def neg_ll_p1(beta):
    b0, b = beta[0], beta[1:]
    nll, grad = 0.0, np.zeros_like(beta)
    for X, y in groups_p1:
        u = b0 + X @ b
        ld = logsumexp(np.append(0.0, u))
        p = np.exp(u - ld)
        d = p - y
        nll += ld - y @ u
        grad[0] += d.sum()
        grad[1:] += X.T @ d
    return nll, grad

res_p1 = minimize(neg_ll_p1, x0=np.zeros(9), jac=True, method='L-BFGS-B')
beta_p1 = res_p1.x

# Load test data
truth = pd.read_csv('Data/test_ground_truth.csv')
pred = pd.read_csv('Data/test_20_predicted.csv')

y_true = truth['booking_bool'].values
y_agent = pred['booking_bool'].values

# MNL predicted probabilities on test set
test_norm = (pred[feature_cols] - means_p1) / stds_p1

mnl_probs = np.zeros(len(pred))
for srch_id, g in pred.groupby('srch_id'):
    idx = g.index.values
    X = test_norm.loc[idx, feature_cols].values
    u = beta_p1[0] + X @ beta_p1[1:]
    ld = logsumexp(np.append(0.0, u))
    mnl_probs[idx] = np.exp(u - ld)

# Session-level top-1: did the model pick the hotel the customer actually booked?
session_correct_mnl = 0
session_correct_agent = 0
total_sessions = 0
for srch_id, g in pred.groupby('srch_id'):
    idx = g.index.values
    true_y = y_true[idx]
    true_choice = int(idx[np.argmax(true_y)]) if true_y.max() == 1 else None
    agent_y = y_agent[idx]
    agent_choice = int(idx[np.argmax(agent_y)]) if agent_y.max() == 1 else None
    mnl_p = mnl_probs[idx]
    no_purchase_p = 1 - mnl_p.sum()
    mnl_choice = int(idx[np.argmax(mnl_p)]) if mnl_p.max() > no_purchase_p else None
    session_correct_mnl += (mnl_choice == true_choice)
    session_correct_agent += (agent_choice == true_choice)
    total_sessions += 1

top1_mnl = session_correct_mnl / total_sessions
top1_agent = session_correct_agent / total_sessions

print(f'Session top-1 (MNL):   {top1_mnl:.4f}  ({session_correct_mnl}/{total_sessions})')
print(f'Session top-1 (Agent): {top1_agent:.4f}  ({session_correct_agent}/{total_sessions})')

Session top-1 (MNL):   0.2914  (487/1671)
Session top-1 (Agent): 0.1957  (327/1671)


In [22]:
# Behavior comparison: do the models reproduce observed booking patterns?
df = pred.copy()
df['y_true'] = y_true
df['y_agent'] = y_agent
df['p_mnl'] = mnl_probs

def bin_rate(col, bins=None, labels=None):
    d = df.copy()
    if bins is not None:
        d['bin'] = pd.cut(d[col], bins=bins, labels=labels, include_lowest=True)
    else:
        d['bin'] = d[col]
    return d.groupby('bin', observed=False).agg(
        n=('y_true', 'size'),
        real=('y_true', 'mean'),
        agent=('y_agent', 'mean'),
        mnl=('p_mnl', 'mean'),
    )

print('Booking rate by star rating')
print(bin_rate('prop_starrating').round(4), '\n')

print('Booking rate by promotion flag')
print(bin_rate('promotion_flag').round(4), '\n')

q = df['price_usd'].quantile([0, 0.25, 0.5, 0.75, 1.0]).values
print('Booking rate by price quartile')
print(bin_rate('price_usd', bins=q, labels=['Q1 (low)', 'Q2', 'Q3', 'Q4 (high)']).round(4), '\n')

profile_cols = ['prop_starrating', 'prop_review_score', 'price_usd', 'promotion_flag', 'prop_brand_bool']
profile = pd.DataFrame({
    'real_bookings':  df.loc[df['y_true']  == 1, profile_cols].mean(),
    'agent_bookings': df.loc[df['y_agent'] == 1, profile_cols].mean(),
    'mnl_weighted':   {c: np.average(df[c], weights=df['p_mnl']) for c in profile_cols},
})
print('Attribute profile of booked hotels')
print(profile.round(3))


Booking rate by star rating
         n    real   agent     mnl
bin                               
1       93  0.0323  0.0215  0.0201
2     6674  0.0291  0.0288  0.0305
3    12602  0.0398  0.0354  0.0390
4     8545  0.0489  0.0549  0.0449
5     1809  0.0381  0.0315  0.0419 

Booking rate by promotion flag
         n    real   agent     mnl
bin                               
0    25243  0.0362  0.0306  0.0361
1     4480  0.0605  0.0877  0.0552 

Booking rate by price quartile
              n    real   agent     mnl
bin                                    
Q1 (low)   7477  0.0420  0.0522  0.0398
Q2         7405  0.0490  0.0486  0.0435
Q3         7437  0.0395  0.0376  0.0424
Q4 (high)  7404  0.0290  0.0184  0.0301 

Attribute profile of booked hotels
                   real_bookings  agent_bookings  mnl_weighted
prop_starrating            3.300           3.332         3.283
prop_review_score          4.036           4.073         4.047
price_usd                121.984         110.039       